In [2]:
import os
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np 
import seaborn as sns
import scipy.stats as stats
from scipy.stats import f_oneway
# from statannotations.Annotator import Annotator
import matplotlib.patheffects as PathEffects
from statsmodels.graphics.mosaicplot import mosaic

## Read data

### Sensor info

In [3]:
# read sensor id
sensor_info = pd.read_csv('data/SensorID_record_time_data.csv')

In [4]:
sensor_info['id']

0     BL007-02
1     BL007-04
2     BL007-07
3     BL007-09
4     BL007-13
5     BL007-15
6     BL007-17
7     BL007-18
8     BL007-19
9     BL007-22
10    BL007-23
11    BL007-25
12    BL007-31
13    BL007-32
14    BL007-33
15    BL007-34
16    BL007-35
17    BL007-36
18    BL007-39
19    BL007-40
20    BL007-42
21    BL007-43
22    BL007-44
23    BL007-45
24    BL007-46
25    BL007-47
26    BL007-48
27    BL007-49
28    BL007-50
29    BL007-51
30    BL007-52
31    BL007-54
32    BL007-56
33    BL007-57
34    BL007-59
35    BL007-64
36    BL007-66
37    BL007-67
Name: id, dtype: object

In [5]:
# Convert field to datetime type 
sensor_info['battery_charging_start_time'] = pd.to_datetime(sensor_info['battery_charging_start_time'], format="%d/%m/%Y %H:%M")  
sensor_info['battery_charging_end_time'] = pd.to_datetime(sensor_info['battery_charging_end_time'], format="%d/%m/%Y %H:%M")  

sensor_info['recording_start_time_participant'] = pd.to_datetime(sensor_info['recording_start_time_participant'], format="%d/%m/%Y %H:%M")  
sensor_info['recording_end_time_participant'] = pd.to_datetime(sensor_info['recording_end_time_participant'], format="%d/%m/%Y %H:%M")

In [6]:
# Set ID as index
sensor_info = sensor_info.set_index('id')

In [7]:
# Define valid start datetime and end datetime
sensor_info[['start_datetime']] = sensor_info[['battery_charging_start_time']]
sensor_info[['end_datetime']] = sensor_info[['battery_charging_end_time']]

sensor_info.start_datetime.fillna(sensor_info['recording_start_time_participant'], inplace=True)
sensor_info.end_datetime.fillna(sensor_info['recording_end_time_participant'], inplace=True)

C:\Users\jobbo\AppData\Local\Temp\ipykernel_11628\2333481681.py:5: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  sensor_info.start_datetime.fillna(sensor_info['recording_start_time_participant'], inplace=True)
C:\Users\jobbo\AppData\Local\Temp\ipykernel_11628\2333481681.py:6: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are s

# Computational processing

## Time filtering

In [15]:
# Read and filter sensors data
sensors_data_path = 'data/Smart_Citizen_Data_concat/'

sensors_data = {}

for data_path in os.listdir(sensors_data_path):
    df = pd.read_csv(sensors_data_path+data_path)

    # convert TIME field to datetime type
    df['TIME'] = pd.to_datetime(df['TIME'])  
    df['TIME'] = df['TIME'].dt.tz_convert(None)

    # get valid start and end datetime for each sensor
    start_date_value = sensor_info.loc[data_path[:-4]]['start_datetime']
    end_date_value = sensor_info.loc[data_path[:-4]]['end_datetime']

    # filter data based on valid start and end datetime
    df = df[(df['TIME'] >= start_date_value) & (df['TIME'] <= end_date_value)]

    # set datetime as index
    df = df.set_index('TIME')

    # store the filtered data in a dictionary with sensor ID as key
    sensors_data[data_path[:-4]] = df

## Data cleaning

### Check data frequency

In [16]:
# check the frequency of each sensor data
sensors_freq = {}

for index in sensors_data:
    df = sensors_data[index]

    is_continuous = (df.index.to_series().diff().iloc[1:] == pd.Timedelta(minutes=1)).all()

    sensors_freq[index] = is_continuous

# find data gaps 
sensor_gaps = {}

for index in sensors_data:
    df = sensors_data[index]
    time_diff = df.index.to_series().diff()

    # Find gaps larger than 1 minute
    gaps = time_diff[time_diff > pd.Timedelta(minutes=1)]

    sensor_gaps[index] = gaps

In [17]:
sensor_gaps

{'BL007-02': TIME
 2023-08-12 15:39:37   0 days 15:40:26
 2023-08-20 03:00:06   0 days 03:00:29
 2023-08-25 06:57:02   0 days 06:57:56
 2023-08-28 10:06:45   0 days 10:07:43
 2023-09-04 03:00:05   0 days 03:00:20
 2023-09-11 03:00:06   0 days 03:01:01
 Name: TIME, dtype: timedelta64[ns],
 'BL007-04': TIME
 2023-08-02 18:45:40   0 days 00:01:06
 2023-08-09 03:00:43   0 days 00:02:03
 2023-08-15 03:31:03   0 days 00:01:20
 2023-08-21 03:00:06   0 days 00:02:03
 2023-08-27 03:30:21   0 days 00:01:15
 2023-09-05 03:00:56   0 days 00:02:03
 2023-09-11 03:29:55   0 days 00:01:59
 Name: TIME, dtype: timedelta64[ns],
 'BL007-07': TIME
 2023-08-10 11:25:20   0 days 11:25:31
 2023-08-17 00:41:54   0 days 00:42:34
 2023-08-24 03:00:05   0 days 03:00:11
 2023-08-28 12:55:35   0 days 12:56:30
 2023-09-05 03:00:02   0 days 03:00:27
 2023-09-12 03:00:03   0 days 03:01:01
 Name: TIME, dtype: timedelta64[ns],
 'BL007-09': TIME
 2023-08-12 03:00:03   0 days 03:00:56
 2023-08-19 03:00:03   0 days 03:01:0

### Check missing data

In [10]:
# check missing data at one minute interval
for index in sensors_data:
    df = sensors_data[index]

    print(f"Number of missing values in {index}: {df.isna().sum()}")

Number of missing values in BL007-02: Unnamed: 0         0
TEMP               0
HUM                0
BATT               0
LIGHT              0
NOISE_A           27
PRESS              0
CCS811_VOCS       32
CCS811_ECO2       32
PM_1           43971
PM_25          43971
PM_10          43971
dtype: int64
Number of missing values in BL007-04: Unnamed: 0         0
TEMP               0
HUM                0
BATT               0
LIGHT              0
NOISE_A           91
PRESS              0
CCS811_VOCS       43
CCS811_ECO2       43
PM_1           56270
PM_25          56270
PM_10          56270
dtype: int64
Number of missing values in BL007-07: Unnamed: 0         0
TEMP               0
HUM                0
BATT               0
LIGHT              0
NOISE_A           20
PRESS              0
CCS811_VOCS       33
CCS811_ECO2       34
PM_1           45422
PM_25          45422
PM_10          45422
dtype: int64
Number of missing values in BL007-09: Unnamed: 0         0
TEMP               0
HUM        

In [11]:
# aggregate missing counts and total counts per variable across all sensors
missing_counts = None
total_counts = None

for index in sensors_data:
    df = sensors_data[index]
    
    if missing_counts is None:
        missing_counts = df.isna().sum()
        total_counts = df.shape[0] * pd.Series(1, index=df.columns)
    else:
        missing_counts = missing_counts.add(df.isna().sum(), fill_value=0)
        total_counts = total_counts.add(df.shape[0] * pd.Series(1, index=df.columns), fill_value=0)

missing_pct = (missing_counts / total_counts) * 100

print("Percentage of missing values per variable (across all sensors):")
print(missing_pct.sort_values(ascending=False).round(2))

Percentage of missing values per variable (across all sensors):
PM_10          80.00
PM_1           80.00
PM_25          80.00
NOISE_A         2.95
CCS811_VOCS     0.06
CCS811_ECO2     0.06
TEMP            0.00
Unnamed: 0      0.00
BATT            0.00
HUM             0.00
PRESS           0.00
LIGHT           0.00
dtype: float64


In [11]:
# check missing data at 5-minute interval columns
missing_data_5min = {}

for index in sensors_data:
    df = sensors_data[index]

    five_min_cols = ['PM_1','PM_25','PM_10']

    # if the number of non-missing values multiplied by 5 minus the total number of rows is greater than or equal to -1, then it means that there are at most 1 missing value in the 5-minute interval columns which is acceptable 
    missing_data_5min[index] = ((df.shape[0]-df[five_min_cols].isna().sum())*5 - df.shape[0]) >= -1

In [12]:
missing_data_5min

{'BL007-02': PM_1     True
 PM_25    True
 PM_10    True
 dtype: bool,
 'BL007-04': PM_1     True
 PM_25    True
 PM_10    True
 dtype: bool,
 'BL007-07': PM_1     True
 PM_25    True
 PM_10    True
 dtype: bool,
 'BL007-09': PM_1     True
 PM_25    True
 PM_10    True
 dtype: bool,
 'BL007-13': PM_1     True
 PM_25    True
 PM_10    True
 dtype: bool,
 'BL007-15': PM_1     True
 PM_25    True
 PM_10    True
 dtype: bool,
 'BL007-17': PM_1     True
 PM_25    True
 PM_10    True
 dtype: bool,
 'BL007-18': PM_1     True
 PM_25    True
 PM_10    True
 dtype: bool,
 'BL007-19': PM_1     True
 PM_25    True
 PM_10    True
 dtype: bool,
 'BL007-22': PM_1     True
 PM_25    True
 PM_10    True
 dtype: bool,
 'BL007-23': PM_1     True
 PM_25    True
 PM_10    True
 dtype: bool,
 'BL007-25': PM_1     True
 PM_25    True
 PM_10    True
 dtype: bool,
 'BL007-31': PM_1     True
 PM_25    True
 PM_10    True
 dtype: bool,
 'BL007-32': PM_1     True
 PM_25    True
 PM_10    True
 dtype: bool,
 'BL00

### Check valid collection period

In [22]:
col_period = {}

for index in sensors_data:
    df = sensors_data[index]

    col_period[index] = df.index.max() - df.index.min()

In [23]:
col_period

{'BL007-02': Timedelta('39 days 21:46:55'),
 'BL007-04': Timedelta('48 days 20:24:21'),
 'BL007-07': Timedelta('40 days 20:17:14'),
 'BL007-09': Timedelta('40 days 04:37:56'),
 'BL007-13': Timedelta('45 days 21:39:58'),
 'BL007-15': Timedelta('47 days 21:36:36'),
 'BL007-17': Timedelta('38 days 17:33:28'),
 'BL007-18': Timedelta('39 days 04:48:02'),
 'BL007-19': Timedelta('48 days 17:37:31'),
 'BL007-22': Timedelta('46 days 11:38:38'),
 'BL007-23': Timedelta('36 days 00:11:08'),
 'BL007-25': Timedelta('38 days 07:22:07'),
 'BL007-31': Timedelta('52 days 21:58:28'),
 'BL007-32': Timedelta('45 days 05:09:17'),
 'BL007-33': Timedelta('43 days 17:33:31'),
 'BL007-34': Timedelta('48 days 05:05:45'),
 'BL007-35': Timedelta('42 days 20:43:12'),
 'BL007-36': Timedelta('39 days 21:55:37'),
 'BL007-39': Timedelta('53 days 03:27:14'),
 'BL007-40': Timedelta('35 days 02:48:18'),
 'BL007-42': Timedelta('36 days 20:46:48'),
 'BL007-43': Timedelta('44 days 16:48:54'),
 'BL007-44': Timedelta('39 days 

# Save data

In [25]:
# create a new directory to save the concatenated files
os.mkdir('data/Smart_Citizen_Data_clean')

# save the concatenated files to the new directory
for index in sensors_data:
    sensors_data[index].to_csv('data/Smart_Citizen_Data_clean/{}.csv'.format(index))